In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

In [0]:
import importlib, utils
importlib.reload(utils)

### Patient

In [0]:
pat_df = spark.sql("select * from health_care.bronze.patient")

In [0]:
pat_obj = utils.custom_trans(pat_df)

pat_df = pat_obj.dedupe(['patient_id'])
pat_df = pat_obj.processing_timestamp()

In [0]:


if spark.catalog.tableExists('health_care.silver.patient'):
  pat_df = pat_obj.upsert('patient',['patient_id'])
else:
  pat_df.write.format('delta')\
      .mode('append')\
      .option('mergeSchema',True)\
      .saveAsTable('health_care.silver.patient')

### Condition

In [0]:
con_df = spark.sql("select * from health_care.bronze.condition")

In [0]:
con_df = con_df.withColumn('condition', lower(col("condition")))
con_df = con_df.withColumn('severity', lower(col("severity")))

con_obj = custom_trans(con_df)
con_df = con_obj.dedupe(['patient_id', 'icd10_code'])
con_df = con_obj.processing_timestamp()


In [0]:
if spark.catalog.tableExists('health_care.silver.condition'):
  con_df = con_obj.upsert('condition',['patient_id', 'icd10_code'])
else:
  con_df.write.format('delta')\
    .mode('append')\
    .option("mergeSchema", "true")\
    .saveAsTable('health_care.silver.condition')

### Medication

In [0]:
med_df = spark.sql("select * from health_care.bronze.medication")

In [0]:
med_df.withColumn('medication', lower(col('medication'))) 

med_obj = custom_trans(med_df)

med_df = med_obj.processing_timestamp()

In [0]:
if spark.catalog.tableExists('health_care.silver.medication'):
  med_df = med_obj.upsert('medication',['patient_id', 'icd10_code'])
else:
  med_df.write.format('delta')\
    .mode('append')\
    .option("mergeSchema", "true")\
    .saveAsTable('health_care.silver.medication')

### Health data

In [0]:
hd_df = spark.sql('select * from health_care.bronze.health_data')

In [0]:
hd_df.withColumn('outcome', lower(col('outcome'))) 

hd_obj = custom_trans(hd_df)

hd_df = hd_obj.processing_timestamp()

In [0]:
if spark.catalog.tableExists('health_care.silver.health_data'):
  hd_df = con_obj.upsert('health_data',['patient_id', 'icd10_code'])
else:
  hd_df.write.format('delta')\
    .mode('append')\
    .option("mergeSchema", "true")\
    .saveAsTable('health_care.silver.health')